# LT Signals Predictivity (Section 5.1)

Evaluates how well Latent-Trajectory (LT) signals predict whether a model's answer is correct, as reported in Section 5.1 (Figure 4 and Table 3) of the paper.

For each model × dataset combination we compute the **ROC-AUC** of each metric against binary answer correctness, then average across combinations to obtain a single summary score per metric.

**Metrics compared:**
- LT signals (Section 3.1): `net_change`, `cumulative_change`, `aligned_change`
- Cross-layer baselines (Wang et al. 2024): `layerwise_mag`, `layerwise_ang`
- Output-distribution baselines (Section 4.1): `logit_margin`, `entropy`, `perplexity`

Saves results to `results/auc_all.csv`.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import torch

import sys
sys.path.insert(0, '../src')
from post_utils import create_internals_df, load_internal_metrics

In [ ]:
PATH = Path('../')  # project root — adjust if running from a different location

## Configuration

In [ ]:
MODELS = [
    'phi4-reasoning-plus',
    'deepseek-r1-qwen14B',
    'qwen3-14B',
    'deepseek-r1-llama8B',
    'deepseek-r1-qwen32B',
    'deepseek-r1-llama70B',
]

DATASETS = ['GPQA', 'AIME2025', 'TSP', 'bigbench/fables', 'bigbench/social-iqa']

# Segment length used to compute internal representations.
# Selects which *_ntokens_{N}_internals.pt file to load.
# Available values: 100, 300, 500, 700
# Note: bigbench and TSP for deepseek-r1-llama8B/qwen32B only have 100 and 300;
#       those combinations fall back to ntokens_300 automatically regardless of this setting.
AVERAGE_TYPE = 'ntokens_500'

# All 8 paper metrics
ALL_METRICS = [
    'net_change',        # LT signal
    'cumulative_change', # LT signal
    'aligned_change',    # LT signal
    'layerwise_mag',     # Wang et al. baseline
    'layerwise_ang',     # Wang et al. baseline
    'logit_margin',      # output-distribution baseline (requires early exit)
    'entropy',           # output-distribution baseline (requires early exit)
    'perplexity',        # output-distribution baseline (requires early exit)
]

# Metrics where higher value predicts lower accuracy — negate before AUC
NEGATE = {'cumulative_change', 'layerwise_ang'}

# Human-readable labels for display
LABEL_MAP = {
    'net_change':        'Net Change',
    'cumulative_change': 'Cumulative Change',
    'aligned_change':    'Aligned Change',
    'layerwise_mag':     'Layer Mag',
    'layerwise_ang':     'Layer Ang',
    'logit_margin':      'Logit Margin',
    'entropy':           'Entropy',
    'perplexity':        'Perplexity',
}

## Data loading

Two loading paths:
- **With early exit** (`create_internals_df`): used when `compute_early_exit_properties.py` was run for this model/dataset. Provides all 8 metrics.
- **Without early exit** (direct `.pt` load).

In [ ]:
def load_with_early_exit(model_name, dataset):
    """Load internals + early-exit data. Returns a single concatenated DataFrame."""
    internals_df, early_exit_df = create_internals_df(
        model_name=model_name,
        dataset=dataset,
        path=PATH,
        average_type=AVERAGE_TYPE,
    )
    internals_df = internals_df.groupby(
        ['datapoint', 'data_repeat', 'metric_name']
    ).mean(numeric_only=True).reset_index()
    internals_df['accuracy'] = internals_df['accuracy'] > 0
    return pd.concat([internals_df, early_exit_df], ignore_index=True), ALL_METRICS


def _load_acc_map(model_name, dataset):
    """Load accuracy from an _only_final early-exit file.

    Returns {(data_point_id, data_repeat_id): acc_at_100pct}.
    Used when the internals .pt file was produced without storing accuracy.
    """
    f = PATH / f"results/early_exit/{dataset}/{model_name}_output_properties_only_final.json"
    with open(f) as fh:
        df = pd.DataFrame(json.load(fh))
    return dict(zip(zip(df['data_point_id'], df['data_repeat_id']), df['acc_100']))


def load_without_early_exit(model_name, dataset, average_type=None):
    """Load internals only; accuracy is read from the .pt file or a fallback acc file."""
    if average_type is None:
        average_type = AVERAGE_TYPE
    lt_wang_metrics = [
        'net_change', 'cumulative_change', 'aligned_change',
        'layerwise_mag', 'layerwise_ang',
    ]
    trajectory_metrics = {'layerwise_mag', 'layerwise_ang', 'seqwise_mag', 'seqwise_ang'}

    internals = load_internal_metrics(model_name, dataset, PATH, average_type=average_type)

    # Some internals files don't store accuracy; fall back to the _only_final early-exit file.
    acc_map = {}
    if internals and 'accuracy' not in internals[0]:
        acc_map = _load_acc_map(model_name, dataset)

    rows = []
    for i in internals:
        acc = i['accuracy'] if 'accuracy' in i else acc_map.get(
            (i['data_point_id'], i['data_repeat_id']), 0
        )
        base = {
            'datapoint':      i['data_point_id'],
            'data_repeat':    i['data_repeat_id'],
            'accuracy':       acc > 0,
            'n_think_tokens': i['n_think_tokens'],
        }
        for metric in lt_wang_metrics:
            if metric in trajectory_metrics:
                rows.append({**base, 'metric_name': metric, 'layer': 'all',
                             'value': torch.mean(i[metric]).item()})
            else:
                for layer_idx, val in enumerate(i[metric]):
                    rows.append({**base, 'metric_name': metric, 'layer': layer_idx, 'value': val})

    df = pd.DataFrame(rows)
    df = df.groupby(['datapoint', 'data_repeat', 'metric_name']).mean(numeric_only=True).reset_index()
    df['accuracy'] = df['accuracy'] > 0
    return df, lt_wang_metrics


# Models that only have ntokens_300 results (no ntokens_500)
_NTOKENS_300_MODELS = {'deepseek-r1-llama8B', 'deepseek-r1-qwen32B'}

# Models without a full per-percentage early-exit file
_NO_EARLY_EXIT_MODELS = {'deepseek-r1-llama70B', 'deepseek-r1-llama8B', 'deepseek-r1-qwen32B'}


def load_data(model_name, dataset):
    """Select loading path and average_type based on data availability."""
    no_early_exit = dataset.startswith('bigbench') or model_name in _NO_EARLY_EXIT_MODELS

    # bigbench and some TSP combos only have ntokens_300
    use_ntokens_300 = (
        dataset.startswith('bigbench')
        or (dataset == 'TSP' and model_name in _NTOKENS_300_MODELS)
    )
    average_type = 'ntokens_300' if use_ntokens_300 else AVERAGE_TYPE

    if no_early_exit:
        return load_without_early_exit(model_name, dataset, average_type=average_type)
    return load_with_early_exit(model_name, dataset)

## AUC computation

For each (model, dataset, metric) triple, computes ROC-AUC of the metric score against binary correctness. Metrics in `NEGATE` are sign-flipped before scoring (higher raw value → less confident). Entries with a single-class label set or no finite scores are recorded as `nan` and excluded from the summary.

In [ ]:
records = []

for model_name in MODELS:
    for dataset in DATASETS:
        print(model_name, dataset)
        try:
            df, available_metrics = load_data(model_name, dataset)
        except FileNotFoundError as e:
            print(f'  [SKIP] {e}')
            continue

        for metric in available_metrics:
            metric_df = df.loc[df['metric_name'] == metric]
            if metric_df.empty:
                continue

            scores = pd.to_numeric(metric_df['value'], errors='coerce').values
            if metric in NEGATE:
                scores = -scores

            y = metric_df['accuracy'].astype(bool).values
            mask = np.isfinite(scores)
            y_valid, scores_valid = y[mask], scores[mask]

            auc = np.nan
            if mask.sum() == 0:
                print(f'  [WARN] no finite scores: {metric}')
            elif np.unique(y_valid).size < 2:
                print(f'  [WARN] single class: {metric}')
            else:
                try:
                    auc = roc_auc_score(y_valid, scores_valid)
                except Exception as e:
                    print(f'  [WARN] AUC error for {metric}: {e}')

            records.append({
                'model_name': model_name,
                'dataset':    dataset,
                'metric':     metric,
                'auc':        None if np.isnan(auc) else round(float(auc), 4),
            })

auc_df = pd.DataFrame(records)

In [ ]:
output_path = PATH / 'results' / 'auc_all.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
auc_df.to_csv(output_path, index=False)
print(f'Saved to {output_path}')

## Summary

Mean and std of ROC-AUC across all (model, dataset) pairs. Corresponds to Table 3 in the paper. LT signals (Net Change, Cumulative Change, Aligned Change) consistently outperform both the cross-layer baselines from Wang et al. (2024) and the output-distribution baselines, demonstrating that the trajectory of hidden states in representation space is a more reliable predictor of correctness than either layer-wise curvature or output-logit confidence.

In [ ]:
summary = (
    auc_df.groupby('metric')['auc']
    .agg(['mean', 'std'])
    .rename(columns={'mean': 'AUC mean', 'std': 'AUC std'})
    .reindex(ALL_METRICS)
)
summary.index = summary.index.map(LABEL_MAP)
summary.round(3)